# K5 Connection-Class Imbalance

For every K5, measure its raw signed color imbalance (blue edges minus red edges) and the raw signed color imbalance of its exact O1-O4 connection classes. Negative values are red-biased, zero is balanced, and positive values are blue-biased.

A neighbor belongs to exactly one class: a K5 sharing exactly three vertices with the target is counted in O3 only. The complete O1-O4 H0-H10 distributions are retained for each reported K5; the weighted scalar is used only for sorting.

In [ ]:
# Experiment Configuration

RANDOM_SEED = 202_608_084
N_VERTICES = 43
NUMBER_OF_COLORINGS = 1
TOP_K5S = 20

# Relative weights of the normalized O1, O2, O3, O4 means.
# These are deliberately configurable: they are an experimental choice,
# not an assumed law of the graph.
CONNECTION_CLASS_WEIGHTS = (1.0, 2.0, 3.0, 4.0)

# True reports the largest-magnitude red- or blue-biased neighborhoods.
# False reports the most positive (blue-biased) neighborhoods only.
SORT_BY_ABSOLUTE_IMBALANCE = True

In [ ]:
# Imports and Graph Setup

from time import perf_counter

import numpy as np
import pandas as pd

from ramsey import (
    RGraph,
    RProblem,
    RRandomConstruction,
    RSearchState,
    calculate_k5_connection_analysis,
)

rng = np.random.default_rng(RANDOM_SEED)
graph = RGraph(RProblem.r55(n_vertices=N_VERTICES))
construction = RRandomConstruction(rng)

print("Vertices:", N_VERTICES)
print("Edges:", f"{graph.number_of_edges:,}")
print("K5s:", f"{graph.subgraph_index(5).clique_count:,}")
print("Colorings:", NUMBER_OF_COLORINGS)
print("Class weights O1-O4:", CONNECTION_CLASS_WEIGHTS)

In [ ]:
# Analyze the Random Coloring Batch

report_rows = []

# Complete H0-H10 distributions for every O class of every reported K5.
# Key: (coloring_number, k5_index). Value shape: (4, 11).
top_connection_histograms = {}

experiment_start = perf_counter()

for coloring_number in range(NUMBER_OF_COLORINGS):
    coloring = construction.construct(graph)
    state = RSearchState(coloring)

    analysis_start = perf_counter()
    analysis = calculate_k5_connection_analysis(
        state,
        class_weights=CONNECTION_CLASS_WEIGHTS,
    )
    analysis_elapsed = perf_counter() - analysis_start

    weighted = analysis.weighted_connection_imbalances

    if SORT_BY_ABSOLUTE_IMBALANCE:
        sort_values = np.abs(weighted)
    else:
        sort_values = weighted

    top_indices = np.argsort(sort_values)[-TOP_K5S:][::-1]

    for rank, k5_index_value in enumerate(top_indices, start=1):
        k5_index = int(k5_index_value)
        local = int(analysis.local_imbalances[k5_index])
        class_means = analysis.class_mean_imbalances[k5_index]

        connection_histograms = analysis.connection_histograms(
            state,
            k5_index,
        )

        # Independently recover each class mean from its complete H0-H10
        # distribution. This also verifies exact, one-class-only counting.
        signed_bins = np.arange(-10, 11, 2, dtype=np.float64)
        histogram_means = (
            connection_histograms @ signed_bins
        ) / analysis.class_sizes

        if not np.allclose(histogram_means, class_means):
            raise RuntimeError(
                f"O-class histogram means disagree for K5 {k5_index}."
            )

        if not np.array_equal(
            connection_histograms.sum(axis=1),
            analysis.class_sizes,
        ):
            raise RuntimeError(
                f"O-class neighbor counts disagree for K5 {k5_index}."
            )

        top_connection_histograms[(coloring_number, k5_index)] = (
            connection_histograms.copy()
        )

        blue_edges = (local + 10) // 2
        red_edges = 10 - blue_edges

        report_rows.append(
            {
                "coloring": coloring_number,
                "rank": rank,
                "k5_index": k5_index,
                "vertices": tuple(
                    int(value)
                    for value in analysis.clique_vertices[k5_index]
                ),
                "B/R": f"{blue_edges}/{red_edges}",
                "local": local,
                "O1": float(class_means[0]),
                "O2": float(class_means[1]),
                "O3": float(class_means[2]),
                "O4": float(class_means[3]),
                "weighted_overlap": float(weighted[k5_index]),
            }
        )

    print(
        f"Coloring {coloring_number + 1}/{NUMBER_OF_COLORINGS} | "
        f"S={state.score:,} | "
        f"analysis={analysis_elapsed:.3f}s"
    )

experiment_elapsed = perf_counter() - experiment_start

report = pd.DataFrame(report_rows)

print()
print("Elapsed:", f"{experiment_elapsed:.3f}s")

In [ ]:
# Top K5 Connection-Class Report

display_report = report.copy()

for column in (
    "O1",
    "O2",
    "O3",
    "O4",
    "weighted_overlap",
):
    display_report[column] = display_report[column].map(
        lambda value: f"{value:+.4f}"
    )

display(display_report)

print()
print("Interpretation:")
print("  local < 0: target K5 is red-biased")
print("  local > 0: target K5 is blue-biased")
print("  O1..O4 < 0: that exact connection class is red-biased")
print("  O1..O4 > 0: that exact connection class is blue-biased")
print("  weighted_overlap is neighborhood-only; target local is not used")

In [ ]:
# Inspect the Complete O1-O4 Distribution of the Top-Ranked K5

top_row = report.iloc[0]
key = (int(top_row["coloring"]), int(top_row["k5_index"]))
histograms = top_connection_histograms[key]

full_distribution = pd.DataFrame(
    histograms,
    index=["O1", "O2", "O3", "O4"],
    columns=[f"H{index}" for index in range(11)],
)

print("K5 index:", key[1])
print("Vertices:", top_row["vertices"])
display(full_distribution)